# 02_playlists

DML: bronze_playlists — Raw playlist metadata.

In [ ]:
%run ../../tools/config/settings

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

raw_path = raw_base_path("playlists", ingestion_date, run_id)
raw_df   = spark.read.json(f"{raw_path}/*.json")

bronze = (
    raw_df
    .select(F.explode("items").alias("item"))
    .select(
        F.col("item.id").alias("playlist_id"),
        F.col("item.name").alias("playlist_name"),
        F.col("item.description").alias("description"),
        F.col("item.owner.id").alias("owner_id"),
        F.col("item.owner.display_name").alias("owner_name"),
        F.col("item.tracks.total").alias("total_tracks"),
        F.col("item.public").alias("is_public"),
        F.col("item.collaborative").alias("is_collaborative"),
        F.to_json(F.col("item")).alias("_raw"),
    )
    .withColumn("run_id",         F.lit(run_id))
    .withColumn("ingestion_date", F.to_date(F.lit(ingestion_date)))
)

bronze.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_playlists")
print(f"bronze_playlists: {bronze.count()} rows written")